# 🚀 Notebook 2: Database Optimization

Before adding caching or replicas, optimize your existing database. Proper indexing solves most read scaling problems.

## Learning Objectives

By the end of this notebook, you'll understand:
- How indexes work (B-tree, Hash)
- Creating effective indexes
- Composite indexes and column order
- Reading EXPLAIN output

---

🔍 **Open Adminer** at http://localhost:8080 to run queries and see execution plans!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [ ]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    try:
        results = cursor.fetchall()
    except:
        results = []
    conn.commit()
    conn.close()
    return results

def measure_query(query: str) -> tuple:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()
    return elapsed, len(results)

def explain_query(query: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    plan = [row[0] for row in cursor.fetchall()]
    conn.close()
    return plan

print("✅ Connected to PostgreSQL")

## 📚 How Indexes Work

An index is like a book's index - instead of reading every page, you look up where to find what you need.

In [ ]:
print("📚 Index Types")
print("=" * 60)
print("""
B-TREE INDEX (Default)
─────────────────────────────────────────────────────────────
• Best for: Range queries, equality, sorting
• Supports: <, <=, =, >=, >, BETWEEN, LIKE 'prefix%'
• Structure: Balanced tree with O(log n) lookups

              [50]
             /    \\
         [25]      [75]
        /    \\    /    \\
     [10]  [30] [60]  [90]

Example: Finding email = 'user50@example.com'
         Only checks ~3-4 nodes instead of 100 rows!

─────────────────────────────────────────────────────────────

HASH INDEX
─────────────────────────────────────────────────────────────
• Best for: Exact equality only (=)
• Does NOT support: Range queries, sorting
• Structure: Hash table with O(1) lookups

hash('user50@example.com') → bucket[42] → row pointer

─────────────────────────────────────────────────────────────
""")

print("💡 Use B-tree (default) unless you ONLY do exact matches!")

## 🔬 Before and After: Index Impact

In [ ]:
# Start from a clean slate: drop BOTH the plain and covering email indexes.
# (A later cell in this notebook creates the covering index — on re-runs, it
# would still exist and fool the "no index" measurement into looking fast.)
run_query("DROP INDEX IF EXISTS idx_users_email")
run_query("DROP INDEX IF EXISTS idx_users_email_covering")

print("🔬 BEFORE INDEX: Find user by email")
print("=" * 60)

query = "SELECT * FROM users WHERE email = 'user50@example.com'"

# Best-of-3: one sample on a busy laptop is noise, not a measurement.
seq_ms = min(measure_query(query)[0] for _ in range(3))
_, count = measure_query(query)
plan_before = "\n".join(explain_query(query))

print(f"\nTime: {seq_ms:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in plan_before.split("\n"):
    print(f"  {line}")

# The whole before/after comparison is meaningless unless Postgres really is
# reading every row here. Fail loudly rather than quietly measure nothing.
assert "Seq Scan" in plan_before, (
    "expected a Seq Scan with no index on users.email — some other index is "
    f"still helping this query. Plan was:\n{plan_before}"
)
print("\n💡 'Seq Scan' — Postgres read all 20,000 rows to find one.")

In [ ]:
print("🔨 Creating index on users.email...")
run_query("CREATE INDEX idx_users_email ON users(email)")
print("✅ Index created!\n")

print("🔬 AFTER INDEX: Find user by email")
print("=" * 60)

idx_ms = min(measure_query(query)[0] for _ in range(3))
_, count = measure_query(query)
plan_after = "\n".join(explain_query(query))

print(f"\nTime: {idx_ms:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in plan_after.split("\n"):
    print(f"  {line}")

assert "Index Scan" in plan_after, (
    f"Postgres did not use the new index. Plan was:\n{plan_after}"
)
assert "Seq Scan" not in plan_after, (
    f"the plan still contains a Seq Scan:\n{plan_after}"
)
assert idx_ms < seq_ms, (
    f"the index made the query slower?! {idx_ms:.2f}ms vs {seq_ms:.2f}ms"
)

print(f"\n💡 'Index Scan' instead of 'Seq Scan' — and {seq_ms / idx_ms:.1f}x faster.")
print( "   Note the speedup is bounded by the network round trip: the QUERY got")
print( "   ~100x cheaper, but the client still pays a millisecond to ask.")

## 🎯 Composite Indexes

When queries filter on multiple columns, composite indexes help.

In [ ]:
print("🎯 Composite Index: Products by category and price")
print("=" * 60)

query = '''
SELECT id, name, price FROM products
WHERE category = 'Electronics' AND price < 100
ORDER BY price
LIMIT 20
'''

# Make sure no helpful index is lingering.
run_query("DROP INDEX IF EXISTS idx_products_category_price")
run_query("DROP INDEX IF EXISTS idx_products_category")

print("\nBEFORE composite index — Postgres must Seq Scan + sort:")
elapsed, count = measure_query(query)
print(f"  Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:4]:
    print(f"  {line}")


In [ ]:
print("\n🔨 Creating composite index (category, price)...")
run_query("CREATE INDEX idx_products_category_price ON products(category, price)")

query = '''
SELECT id, name, price FROM products
WHERE category = 'Electronics' AND price < 100
ORDER BY price
LIMIT 20
'''

print("\nAFTER composite index — Postgres jumps straight to matching rows:")
elapsed, count = measure_query(query)
print(f"  Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:4]:
    print(f"  {line}")

print("\n💡 Plan should now mention 'Index Scan using idx_products_category_price'.")


## ⚠️ Column Order Matters!

In [ ]:
print("⚠️ Composite Index Column Order")
print("=" * 60)
print("""
Index on (category, price) supports:
─────────────────────────────────────────────────────────────
✅ WHERE category = 'X'                    (leftmost column)
✅ WHERE category = 'X' AND price < 100    (both columns)
❌ WHERE price < 100                       (skips leftmost!)

Think of it like a phone book:
─────────────────────────────────────────────────────────────
• Sorted by (LastName, FirstName)
• Easy to find all "Smith"s
• Easy to find "Smith, John"
• Hard to find all "John"s (scattered throughout!)
""")

print("\n🔬 Query using ONLY price (skips category):")
query_price_only = "SELECT * FROM products WHERE price < 50"
plan_price = "\n".join(explain_query(query_price_only))
for line in plan_price.split("\n")[:2]:
    print(f"  {line}")

# Say what actually happened rather than hard-coding the answer we hoped for.
if "Seq Scan" in plan_price:
    print("\n💡 Seq Scan — the (category, price) index is no help. Rows for a given")
    print("   price are scattered across every category, exactly like all the")
    print("   \"John\"s scattered through a phone book sorted by surname.")
else:
    print("\n💡 Postgres reached for the index anyway — it CAN read a composite index")
    print("   end to end and filter on the second column. But look at the cost: that")
    print("   is a full index scan, not a seek. Still O(index), not O(log n).")

## 📊 Index Recommendations

In [ ]:
print("📊 When to Create Indexes")
print("=" * 60)
print("""
✅ DO INDEX:
─────────────────────────────────────────────────────────────
• Primary keys (automatic)
• Foreign keys used in JOINs
• Columns in WHERE clauses
• Columns in ORDER BY
• Columns with high cardinality (many unique values)

❌ DON'T INDEX:
─────────────────────────────────────────────────────────────
• Tiny tables (< 1000 rows)
• Columns with low cardinality (e.g., boolean, status)
• Columns rarely used in queries
• Tables with heavy writes and few reads

⚖️ TRADE-OFFS:
─────────────────────────────────────────────────────────────
• Indexes speed up reads but slow down writes
• Each index adds storage overhead
• Too many indexes → slower INSERT/UPDATE/DELETE
• For read-heavy apps, index liberally!
""")

In [ ]:
print("📋 Creating Common Indexes for Our Schema")
print("=" * 60)

indexes = [
    ("idx_posts_user_id", "posts(user_id)", "Find posts by author"),
    ("idx_posts_created_at", "posts(created_at DESC)", "Recent posts"),
    ("idx_comments_post_id", "comments(post_id)", "Comments on a post"),
    ("idx_reviews_product_id", "reviews(product_id)", "Reviews for product"),
    ("idx_short_urls_code", "short_urls(short_code)", "URL lookup"),
]

for idx_name, idx_def, description in indexes:
    run_query(f"DROP INDEX IF EXISTS {idx_name}")
    run_query(f"CREATE INDEX {idx_name} ON {idx_def}")
    print(f"✅ {idx_name}: {description}")

print("\n💡 These indexes will help all our subsequent queries!")

## 🔍 Query Optimization Tips

In [ ]:
print("🔍 Query Optimization Tips")
print("=" * 60)
print("""
1. SELECT ONLY WHAT YOU NEED
─────────────────────────────────────────────────────────────
❌ SELECT * FROM users WHERE id = 1
✅ SELECT username, email FROM users WHERE id = 1

2. USE LIMIT FOR PAGINATION
─────────────────────────────────────────────────────────────
❌ SELECT * FROM posts ORDER BY created_at DESC
✅ SELECT * FROM posts ORDER BY created_at DESC LIMIT 20

3. AVOID FUNCTIONS ON INDEXED COLUMNS
─────────────────────────────────────────────────────────────
❌ WHERE LOWER(email) = 'user@example.com'  -- Can't use index
✅ WHERE email = 'user@example.com'          -- Uses index

4. USE EXISTS INSTEAD OF COUNT FOR EXISTENCE CHECKS
─────────────────────────────────────────────────────────────
❌ SELECT COUNT(*) FROM likes WHERE post_id = 1  -- Scans all
✅ SELECT EXISTS(SELECT 1 FROM likes WHERE post_id = 1)  -- Stops early

5. BATCH QUERIES WHEN POSSIBLE
─────────────────────────────────────────────────────────────
❌ Loop: SELECT * FROM users WHERE id = 1, 2, 3...
✅ SELECT * FROM users WHERE id IN (1, 2, 3, ...)
""")

## 🧪 Quick Quiz

1. **What's the difference between B-tree and Hash indexes?**

2. **For index (A, B, C), which queries can use it?**

3. **Why not index every column?**

## 🎯 Covering Indexes (PostgreSQL `INCLUDE`)

A **covering index** stores extra columns inside the index itself so the
database can answer the whole query from the index — without touching the
table. This is sometimes called an "index-only scan".

```
Normal index on (email):
   lookup email → get row pointer → fetch row from table  (2 steps)

Covering index on (email) INCLUDE (username, display_name):
   lookup email → index already has username + display_name  (1 step)
```

**When to use**: hot read paths where you always select the same few columns.
Classic example: authentication (look up user by email, only need id + password_hash).


In [ ]:
# Demo: covering index on users(email) INCLUDE (username, display_name)

query = "SELECT username, display_name FROM users WHERE email = 'user500@example.com'"

run_query("DROP INDEX IF EXISTS idx_users_email")
run_query("DROP INDEX IF EXISTS idx_users_email_covering")

# "Index Only Scan" is a promise Postgres can only keep if the VISIBILITY MAP
# says a page is all-visible — and only VACUUM maintains that map. Straight
# after a bulk load (or after notebook 5 updated a row) the map is stale, the
# scan has to visit the heap for every row anyway, and EXPLAIN reports
# `Heap Fetches: N`. So vacuum first, then the claim below is actually true.
vac = psycopg2.connect(**DB_CONFIG)
vac.autocommit = True                      # VACUUM cannot run inside a transaction
vac.cursor().execute("VACUUM ANALYZE users")
vac.close()

# Plain index
run_query("CREATE INDEX idx_users_email ON users(email)")
print("Plain index on (email):")
plan_plain = "\n".join(explain_query(query))
for line in plan_plain.split("\n")[:4]:
    print(f"  {line}")
print(f"  Time: {min(measure_query(query)[0] for _ in range(3)):.2f}ms\n")

# Covering index — the engine can answer from the index alone
run_query("DROP INDEX idx_users_email")
run_query(
    "CREATE INDEX idx_users_email_covering "
    "ON users(email) INCLUDE (username, display_name)"
)
print("Covering index on (email) INCLUDE (username, display_name):")
plan_cov = "\n".join(explain_query(query))
for line in plan_cov.split("\n")[:4]:
    print(f"  {line}")
print(f"  Time: {min(measure_query(query)[0] for _ in range(3)):.2f}ms")

assert "Index Only Scan" in plan_cov, (
    f"the covering index did not produce an Index Only Scan:\n{plan_cov}"
)

# "Zero table reads" is a testable claim, so test it.
heap_lines = [l.strip() for l in plan_cov.split("\n") if "Heap Fetches" in l]
heap_fetches = int(heap_lines[0].split(":")[1]) if heap_lines else 0
print(f"\n  {heap_lines[0] if heap_lines else 'Heap Fetches: (not reported)'}")
assert heap_fetches == 0, (
    f"Index Only Scan still went to the heap {heap_fetches} time(s) — the "
    f"visibility map is stale, so this is NOT the zero-table-read win the "
    f"markdown above advertises. Re-run VACUUM ANALYZE users."
)

print("\n💡 'Index Only Scan' with Heap Fetches: 0 — the table was never touched.")
print("   That last part is the whole point, and it is NOT free: it depends on")
print("   VACUUM keeping the visibility map current. On a heavily-updated table")
print("   with lazy autovacuum, your 'index only' scan quietly starts reading")
print("   the heap again and the win evaporates.")

## 🎯 Partial Indexes (index a SUBSET of rows)

If most queries only care about a small slice of the table, a **partial index**
skips the rest — smaller index, faster updates, cheaper storage.

```sql
-- Only index posts with lots of engagement (the "hot" ones)
CREATE INDEX idx_viral_posts
    ON posts(created_at DESC)
    WHERE like_count > 500;
```

**Real-world examples**:
- `WHERE deleted_at IS NULL` — only index active rows
- `WHERE status = 'pending'` — only index rows the worker queries
- `WHERE is_public = true` — only index publicly visible content


In [ ]:
# Demo: partial index for "viral" posts only
run_query("DROP INDEX IF EXISTS idx_viral_posts")
run_query(
    "CREATE INDEX idx_viral_posts ON posts(created_at DESC) "
    "WHERE like_count > 500"
)

# Check index size — partial index is MUCH smaller than a full index
size = run_query(
    "SELECT pg_size_pretty(pg_relation_size('idx_viral_posts'))"
)
print(f"Partial index size: {size[0][0]}")

query = (
    "SELECT id, content, like_count FROM posts "
    "WHERE like_count > 500 ORDER BY created_at DESC LIMIT 10"
)
print("\nExecution plan (should use idx_viral_posts):")
for line in explain_query(query)[:4]:
    print(f"  {line}")


## 🧭 Specialized Index Types: BRIN & GIN

B-tree covers 90% of cases, but two other index types solve real production problems.

### 📉 BRIN — for huge, naturally ordered tables

**B**lock **R**ange **IN**dex. Instead of one entry per row, it stores the
min/max value for each *block* of rows. Tiny and fast to build — but it only
works if the column is **physically ordered** on disk.

```
Typical use cases:
• Time-series: logs, events, metrics (rows are appended in time order)
• Audit tables:  append-only, never updated
• Sensor data:   billions of rows ordered by timestamp
```

**Why it matters**: a BRIN index on a 100 GB logs table might be ~1 MB. A
B-tree on the same column would be ~10 GB.

**Check before you reach for it**: physical order is measurable, not a vibe —

```sql
SELECT correlation FROM pg_stats
WHERE tablename = 'posts' AND attname = 'created_at';
```

Near **+1.0** and BRIN will prune beautifully. Near **0** and it will still be
tiny, still be built in seconds, and still scan your whole table. Our own
`posts` table is the second case — the next cell shows both side by side.

### 🔎 GIN — for "multi-value" columns

**G**eneralized **IN**verted index. Think of it as a book's back-of-book index:
it maps *each element inside a value* back to the rows containing it.

```
Use GIN when you query things like:
• tags @> ARRAY['python', 'web']      — array containment
• data @> '{"city": "Berlin"}'         — JSONB key/value search
• to_tsvector(body) @@ 'error & 500'   — full-text search
```

**B-tree can't help here** because it sorts whole values, not the pieces inside them.


In [ ]:
# BRIN demo.
#
# ⚠️ Read this before believing the numbers: our `posts` table is a
# COUNTER-EXAMPLE, not an example. init.sql generated created_at as
# `NOW() - random() * interval '30 days'`, so timestamps are scattered
# uniformly across every disk block. BRIN will still be tiny — it is always
# tiny — but it will prune nothing at all. That contrast is the lesson.

run_query("DROP INDEX IF EXISTS idx_posts_created_brin")
run_query("CREATE INDEX idx_posts_created_brin ON posts USING BRIN(created_at)")
# Recreate the B-tree we're comparing against, in case this cell runs standalone.
run_query("CREATE INDEX IF NOT EXISTS idx_posts_created_at ON posts(created_at DESC)")


def index_bytes(name: str) -> int:
    return run_query(f"SELECT pg_relation_size('{name}')")[0][0]


def pretty(n: int) -> str:
    return run_query(f"SELECT pg_size_pretty({n}::bigint)")[0][0]


brin_bytes = index_bytes("idx_posts_created_brin")
btree_bytes = index_bytes("idx_posts_created_at")

print("── SIZE ──────────────────────────────────────────────────────────")
print(f"B-tree index on posts.created_at : {pretty(btree_bytes):>10}")
print(f"BRIN   index on posts.created_at : {pretty(brin_bytes):>10}")
print(f"BRIN is {btree_bytes / brin_bytes:.0f}x smaller on this 50k-row table.")

# Compare the numbers, not the human-readable strings — '1160 kB' vs '48 kB'
# sorts wrong as text, and a size claim nobody checks is a size claim that rots.
assert brin_bytes * 4 < btree_bytes, (
    f"BRIN ({brin_bytes} B) is supposed to be far smaller than the B-tree "
    f"({btree_bytes} B)"
)

# ── But is the tiny index any USE? ────────────────────────────────────────
# pg_stats.correlation is the number BRIN lives or dies by: how closely on-disk
# order matches the column's sort order. +1.0 = perfectly ordered, 0 = random.
#
# We build TWO copies of the same 50k rows — one in the order Postgres already
# has them (scattered), one physically sorted — and give each ONLY a BRIN
# index, so the planner has no other option to fall back on.
for tbl in ("posts_scattered", "posts_timeordered"):
    run_query(f"DROP TABLE IF EXISTS {tbl}")
run_query("CREATE TABLE posts_scattered   AS SELECT * FROM posts")
run_query("CREATE TABLE posts_timeordered AS SELECT * FROM posts ORDER BY created_at")
run_query("CREATE INDEX idx_scattered_brin ON posts_scattered   USING BRIN(created_at)")
run_query("CREATE INDEX idx_ordered_brin   ON posts_timeordered USING BRIN(created_at)")
run_query("ANALYZE posts_scattered")
run_query("ANALYZE posts_timeordered")


def correlation(table: str) -> float:
    return float(run_query(
        f"SELECT correlation FROM pg_stats "
        f"WHERE tablename = '{table}' AND attname = 'created_at'"
    )[0][0])


corr_random = correlation("posts_scattered")
corr_sorted = correlation("posts_timeordered")

print("\n── PHYSICAL ORDER ────────────────────────────────────────────────")
print(f"posts_scattered   correlation = {corr_random:+.3f}   (as init.sql wrote them)")
print(f"posts_timeordered correlation = {corr_sorted:+.3f}   (inserted ORDER BY created_at)")
assert abs(corr_random) < 0.3, (
    f"posts.created_at was supposed to be randomly ordered, correlation is "
    f"{corr_random:+.3f} — the counter-example below no longer works"
)
assert corr_sorted > 0.9, f"the sorted copy should be near +1.0, got {corr_sorted:+.3f}"


def brin_recheck_rows(table: str, index_name: str) -> int:
    """Force the BRIN bitmap scan and report how many rows it had to throw away.

    That number IS the pruning quality: every row BRIN failed to exclude gets
    read from the heap and re-checked one at a time.
    """
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SET enable_seqscan = off")
    cur.execute("SET enable_indexscan = off")
    cur.execute("SET enable_indexonlyscan = off")
    cur.execute(
        f"EXPLAIN ANALYZE SELECT count(*) FROM {table} "
        f"WHERE created_at > NOW() - INTERVAL '1 day'"
    )
    plan = "\n".join(row[0] for row in cur.fetchall())
    conn.close()
    assert index_name in plan, f"expected a BRIN scan on {index_name}:\n{plan}"
    for line in plan.split("\n"):
        if "Rows Removed by Index Recheck" in line:
            return int(line.split(":")[1])
    return 0


removed_random = brin_recheck_rows("posts_scattered", "idx_scattered_brin")
removed_sorted = brin_recheck_rows("posts_timeordered", "idx_ordered_brin")

print("\n── PRUNING (same query, same index type, 1 day out of 30) ────────")
print(f"posts_scattered   : BRIN read and discarded {removed_random:,} rows")
print(f"posts_timeordered : BRIN read and discarded {removed_sorted:,} rows")

assert removed_sorted * 3 < removed_random, (
    f"BRIN on the sorted copy should skip most of the table; discarded "
    f"{removed_sorted:,} vs {removed_random:,} on the scattered one"
)

print()
print("💡 Same index, same size, same query — and on the unsorted table BRIN")
print("   scanned essentially the whole thing. BRIN is a bet on insertion")
print("   order, not a general-purpose index. Check pg_stats.correlation")
print("   BEFORE you reach for it.")
print()
print("💡 When the bet pays off, it pays enormously: BRIN stores one summary")
print("   per block RANGE (O(pages)) where a B-tree stores one entry per ROW")
print("   (O(rows)). On a 100 GB append-only logs table that is ~1 MB vs ~10 GB.")

for tbl in ("posts_scattered", "posts_timeordered"):   # tidy up the demo copies
    run_query(f"DROP TABLE IF EXISTS {tbl}")

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. B-tree vs Hash:")
print("   B-tree: Range queries, sorting, equality")
print("   Hash: Only equality (=), faster for exact match")
print("   Default to B-tree unless you know otherwise")
print()
print("2. Index (A, B, C) supports:")
print("   ✅ WHERE A = x")
print("   ✅ WHERE A = x AND B = y")
print("   ✅ WHERE A = x AND B = y AND C = z")
print("   ❌ WHERE B = y (skips A)")
print("   ❌ WHERE C = z (skips A, B)")
print()
print("3. Why not index everything:")
print("   - Slows down INSERT/UPDATE/DELETE")
print("   - Uses disk space")
print("   - Index maintenance overhead")
print("   - For read-heavy apps, still index liberally!")

## 🔌 Connection Pooling (a real-world must-have)

Every database connection costs memory on the server (~10 MB on Postgres) and
takes ~1-5 ms to open. If every web request opens+closes its own connection:

- At 1,000 requests/sec you spawn 1,000 connection handshakes/sec.
- Postgres caps out around 100-500 connections, then rejects new ones.
- Your app runs fine on a laptop and dies in production.

**Fix**: put a *connection pool* between your app and the database:

| Tool | Where it runs | What it does |
|------|---------------|--------------|
| `psycopg2.pool.SimpleConnectionPool` | In-process (Python) | Reuses connections across requests |
| `SQLAlchemy` engine pool | In-process | Same, with framework integration |
| **PgBouncer** | Separate process | Pools *across* all your app servers (10k client conns → 20 Postgres conns) |
| **RDS Proxy / CloudSQL connectors** | Managed service | Same as PgBouncer, cloud-managed |

### 📐 How big should the pool be? (this is arithmetic, not taste)

Two numbers decide it, and neither of them is "20 feels safe":

1. **Little's Law** — the connections you need *in flight* is
   `concurrency = throughput × service_time`. 2,000 rps of 4 ms queries needs
   **8** connections. Not 8 per server: 8 in total, across the fleet.
2. **Fan-in** — every app server multiplies your pool.
   `total = pool_size × app_servers`, and that total has to stay under
   Postgres's `max_connections` (default **100**).

The counter-intuitive part: **a bigger pool is usually slower.** Postgres runs
one backend process per connection, so once you have more active connections
than the database has cores, you are paying context switches and lock
contention for work the machine cannot do in parallel anyway. The queue has to
live *somewhere* — it is cheaper in your app than inside the database.

The next cell computes all of this instead of asserting it.

**Rule of thumb**: start with in-process pooling (easy), add PgBouncer the
moment `pool_size × app_servers` approaches `max_connections`.


In [ ]:
# Minimal demo: reusing connections via a pool instead of opening new ones per query.
import math
from psycopg2 import pool

pg_pool = pool.SimpleConnectionPool(
    minconn=1, maxconn=5, **DB_CONFIG,
)

def pooled_query(sql: str):
    conn = pg_pool.getconn()          # borrow
    try:
        cur = conn.cursor()
        cur.execute(sql)
        return cur.fetchall()
    finally:
        pg_pool.putconn(conn)          # return, do NOT close

# Compare: 100 queries with new connections vs 100 queries from a pool.
q = "SELECT 1"

start = time.time()
for _ in range(100):
    c = get_connection(); cur = c.cursor(); cur.execute(q); cur.fetchall(); c.close()
new_conn_ms = (time.time() - start) * 1000

start = time.time()
for _ in range(100):
    pooled_query(q)
pooled_ms = (time.time() - start) * 1000

print(f"100 queries, fresh connection each time : {new_conn_ms:7.1f} ms")
print(f"100 queries, pooled connections        : {pooled_ms:7.1f} ms")
print(f"Speedup: {new_conn_ms / pooled_ms:.1f}x")

setup_ms = (new_conn_ms - pooled_ms) / 100
print(f"\n👉 Measured cost of ONE connection handshake: {setup_ms:.2f} ms")

assert pooled_ms < new_conn_ms, (
    f"pooling was not faster than reconnecting ({pooled_ms:.1f}ms vs "
    f"{new_conn_ms:.1f}ms) — the pool is not being reused"
)
pg_pool.closeall()

# ── Now size a real pool, using that measured handshake cost ────────────────
print("\n" + "=" * 68)
print("📐 Sizing a pool for a service that must do 2,000 rps")
print("=" * 68)

target_rps          = 2_000    # what the service must serve
query_ms            = 4.0      # how long a query holds a connection
app_servers         = 8        # pods / processes behind the load balancer
pg_max_connections  = 100      # Postgres default

# Little's Law: concurrency = throughput x service time.
concurrency = target_rps * (query_ms / 1000.0)
per_server = math.ceil(concurrency / app_servers * 1.5)   # +50% burst headroom
sized_total = per_server * app_servers

print(f"\n   Little's Law : {target_rps:,} rps x {query_ms:.0f}ms "
      f"= {concurrency:.0f} connections busy at any instant")
print(f"   Per server   : ceil({concurrency:.0f}/{app_servers} x 1.5) = {per_server}")
print(f"   Fleet total  : {per_server} x {app_servers} = {sized_total} connections "
      f"(max_connections is {pg_max_connections}) ✅")

# The trap: a default that looks harmless per server and is fatal per fleet.
default_pool = 15              # e.g. SQLAlchemy pool_size=5 + max_overflow=10
naive_total = default_pool * app_servers
k8s_total = default_pool * 30  # ...and then someone scales to 30 pods

print(f"\n   ⚠️  Leave the framework default at pool_size={default_pool} instead:")
for label, total in ((f"{app_servers} servers", naive_total), ("30 pods", k8s_total)):
    verdict = "OVER" if total > pg_max_connections else "under"
    print(f"      {default_pool} x {label:<11} = {total:>4} connections "
          f"→ {verdict} the {pg_max_connections} limit")
print(f"\n      Nothing warns you. Postgres simply starts refusing connections,")
print(f"      and it does so under load — i.e. during the incident, not before it.")

assert sized_total <= pg_max_connections, "our own sizing must fit in max_connections"
assert naive_total > pg_max_connections, (
    "this cell's whole point is that the naive default overshoots; if it "
    "doesn't, the numbers above need updating"
)

pgbouncer_backends = math.ceil(concurrency * 1.5)
print(f"\n   🔀 PgBouncer in transaction mode is the fix: {k8s_total} client connections")
print(f"      multiplex onto ~{pgbouncer_backends} real Postgres backends (the fleet only")
print(f"      ever needs {concurrency:.0f} busy at once), because a client")
print(f"      only holds a backend for the duration of a TRANSACTION, not a")
print(f"      whole HTTP request. (Cost: no session state — no LISTEN/NOTIFY,")
print(f"      no session-level temp tables, no cross-transaction prepared statements.)")

## 📚 Summary

### Key Takeaways

1. **Indexes turn O(n) into O(log n)** - massive speedup
2. **Use EXPLAIN** - see how queries actually execute
3. **Column order matters** - leftmost columns used first
4. **Index for your queries** - not generic "best practices"
5. **B-tree is usually right** - handles most use cases

### Next Up

In **Notebook 3**, we'll learn about denormalization:
- Trading storage for speed
- Materialized views
- Pre-computed aggregations